In [8]:
import asyncio
import nest_asyncio
import sys
import threading
import os
import gspread
import re
from oauth2client.service_account import ServiceAccountCredentials
from playwright.async_api import async_playwright
from dotenv import load_dotenv

# 1. Налаштування
load_dotenv()
nest_asyncio.apply()

# Конфігурація Google Sheets
# Переконайтеся, що файл credentials.json лежить у тій же папці
SCOPE = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
CREDENTIALS_FILE = 'credentials.json'
SPREADSHEET_ID = os.getenv('SPREADSHEET_ID')

def get_sheet():
    """Авторизація та підключення до таблиці"""
    creds = ServiceAccountCredentials.from_json_keyfile_name(CREDENTIALS_FILE, SCOPE)
    client = gspread.authorize(creds)
    return client.open_by_key(SPREADSHEET_ID).sheet1

def clean_text(text, pattern=None):
    """Очищення тексту (наприклад, видалення 'грн', 'м²' тощо)"""
    if not text or text == "Не вказано":
        return ""
    if pattern:
        match = re.search(pattern, text)
        return match.group(1) if match else text
    return text.strip()

async def scrape_and_save():
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    async with async_playwright() as p:
        print("Запуск браузера...")
        browser = await p.chromium.launch(headless=False) # Поставте True, щоб працювало у фоні
        page = await browser.new_page()

        # 2. Отримання списку оголошень
        print("Пошук оголошень на OLX...")
        await page.goto('https://www.olx.ua/uk/nedvizhimost/kvartiry/kiev/', timeout=60000)
        await page.wait_for_load_state('networkidle')

        # Збираємо унікальні посилання
        locators = await page.locator('a[href*="/obyavlenie/"]').all()
        urls = list(set([await l.get_attribute('href') for l in locators]))
        urls = [u if u.startswith('http') else f"https://www.olx.ua{u}" for u in urls]

        # Обмежимо для тесту першими 5 оголошеннями
        urls_to_parse = urls[:5]
        print(f"Знайдено оголошень. Починаємо парсинг {len(urls_to_parse)} штук...")

        final_data = []

        for url in urls_to_parse:
            try:
                print(f"Парсимо: {url}")
                await page.goto(url, timeout=60000)
                await page.wait_for_load_state('domcontentloaded')

                # Витягуємо дані за допомогою стабільних локаторів
                price_loc = page.locator('[data-testid="ad-price"]')
                price_raw = await price_loc.first.inner_text() if await price_loc.count() > 0 else ""

                # Використовуємо .first, щоб уникнути помилки strict mode violation
                area_loc = page.get_by_text("Загальна площа:")
                area_raw = await area_loc.first.inner_text() if await area_loc.count() > 0 else ""

                floor_loc = page.get_by_text("Поверх:")
                floor_raw = await floor_loc.first.inner_text() if await floor_loc.count() > 0 else ""

                total_floors_loc = page.get_by_text("Поверховість:")
                total_floors_raw = await total_floors_loc.first.inner_text() if await total_floors_loc.count() > 0 else ""

                city_loc = page.locator('[data-testid="breadcrumb-item"]')
                city_raw = await city_loc.last.inner_text() if await city_loc.count() > 0 else "Київ"

                # 3. Структурування та очищення
                # Витягуємо лише цифри за допомогою регулярних виразів
                data_row = [
                    url,
                    clean_text(price_raw).replace(' ', ''), # Ціна (видаляємо пробіли)
                    clean_text(floor_raw, r"Поверх:\s*(\d+)"), # Поверх
                    clean_text(total_floors_raw, r"Поверховість:\s*(\d+)"), # Поверховість
                    clean_text(city_raw), # Місто
                    clean_text(area_raw, r"Загальна площа:\s*(\d+)") # Площа
                ]
                final_data.append(data_row)

            except Exception as e:
                print(f"Помилка на сторінці {url}: {e}")

        await browser.close()

        # 4. Запис у Google Sheets
        if final_data:
            print("Запис даних у Google Sheets...")
            sheet = get_sheet()

            # Додаємо заголовки, якщо таблиця порожня
            if not sheet.get_all_values():
                sheet.append_row(["URL", "Ціна", "Поверх", "Поверховість", "Місто", "Площа"])

            sheet.append_rows(final_data)
            print(f"Успішно додано {len(final_data)} рядків!")
        else:
            print("Немає даних для запису.")

# Запуск через потік (як у вашому робочому прикладі)
def run_main():
    asyncio.run(scrape_and_save())

thread = threading.Thread(target=run_main)
thread.start()
thread.join()

Запуск браузера...
Пошук оголошень на OLX...
Знайдено оголошень. Починаємо парсинг 5 штук...
Парсимо: https://www.olx.ua/d/uk/obyavlenie/prodazh-3k-kvartiri-zhk-siretsk-sadi-oselya-derzh-programi-generator-bez-ID107fYx.html?search_reason=search%7Cpromoted
Парсимо: https://www.olx.ua/d/uk/obyavlenie/prodazh-3-km-kvartiri-v-tsentr-blya-metro-zolot-vorota-IDYi7Tb.html?search_reason=search%7Cpromoted
Парсимо: https://www.olx.ua/d/uk/obyavlenie/orenda-kvartiri-v-zhk-frantsuzkiy-kvartal-2-vul-makkeyna-ID10fp3S.html?search_reason=search%7Cpromoted
Парсимо: https://www.olx.ua/d/uk/obyavlenie/dovgostrokova-orenda-suchasno-kvartiri-na-andrvskomu-uzvoz-kontraktova-ploscha-podl-vd-vlasnika-ID10fl4C.html?search_reason=search%7Corganic
Парсимо: https://www.olx.ua/d/uk/obyavlenie/prodazha-kvartiry-dragomirova-11-zhk-novopecherskie-lipki-bez-komissii-IDZ6ppB.html?search_reason=search%7Cpromoted
Запис даних у Google Sheets...
Успішно додано 5 рядків!
